# 🔓 Solutions: Model Generalization

> **Try first, peek second.** You learn far more by struggling with a problem for ten minutes than by reading its answer for ten seconds. Use these solutions to *compare* your approach, not to replace it. There is often more than one correct way.


In [1]:
import sys
sys.path.append("..")
from course_utils import *
setup()

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, learning_curve
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def poly_model(degree, alpha=None):
    """Polynomial regression pipeline: x -> [x, x², ..., x^degree] -> scale -> (Ridge | plain) regression."""
    reg = LinearRegression() if alpha is None else Ridge(alpha=alpha)
    return make_pipeline(PolynomialFeatures(degree, include_bias=False), StandardScaler(), reg)

### 🧩 Problem 5.1: Diagnose the model &nbsp; ⭐

Write `diagnose(train_err, val_err, goal=10)` returning `"underfit"`, `"overfit"` or `"good fit"` using these rules (checked in order):

1. `train_err > goal` → `"underfit"` (can't even fit the training data)
2. else `val_err > 1.5 * train_err` → `"overfit"` (large train/validation gap)
3. otherwise → `"good fit"`

In [2]:
def diagnose(train_err, val_err, goal=10):
    if train_err > goal:
        return "underfit"
    if val_err > 1.5 * train_err:
        return "overfit"
    return "good fit"

print(diagnose(25, 27), diagnose(2, 15), diagnose(8, 9.5))

underfit overfit good fit


In [3]:
# ✔ Self-check
cases = [((25, 27), "underfit"), ((2, 15), "overfit"), ((8, 9.5), "good fit"),
         ((30, 60), "underfit"), ((5, 7), "good fit"), ((4, 6.5), "overfit")]
for args, expected in cases:
    check(f"train={args[0]}, val={args[1]} → {expected}", lambda: diagnose(*args) == expected)

✅ PASSED   train=25, val=27 → underfit
✅ PASSED   train=2, val=15 → overfit
✅ PASSED   train=8, val=9.5 → good fit
✅ PASSED   train=30, val=60 → underfit
✅ PASSED   train=5, val=7 → good fit
✅ PASSED   train=4, val=6.5 → overfit


**🧑‍🏫 Walkthrough.** Order matters: case (30, 60) has a big gap too, but its *training* error is already above target, so the first problem to fix is underfitting. A high-bias model will also look "fine" on the gap criterion, which is why you must check absolute training error, not only the gap.

### 🧩 Problem 5.2: Find the best polynomial degree &nbsp; ⭐⭐

The data is pre-split for you into **train (20), validation (30) and test (30)** points.

1. For every degree from **1 to 12**, fit `poly_model(degree)` on the training set and record the **validation RMSE** in the dictionary `val_errors`.
2. Store the degree with the lowest validation RMSE in `best_degree`.
3. Fit that degree and store its **test RMSE** in `test_error`.

In [4]:
curve = load("noisy_curve.csv")
X_c, y_c = curve[["x"]], curve["y"]
X_train, X_rest, y_train, y_rest = train_test_split(X_c, y_c, train_size=20, random_state=3)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=3)

val_errors = {}
for d in range(1, 13):
    val_errors[d] = rmse(y_val, poly_model(d).fit(X_train, y_train).predict(X_val))

best_degree = min(val_errors, key=val_errors.get)
test_error = rmse(y_test, poly_model(best_degree).fit(X_train, y_train).predict(X_test))
print({d: round(e, 2) for d, e in val_errors.items()})
print(f"best degree = {best_degree}; test RMSE = {test_error:.3f}")

{1: 0.51, 2: 0.62, 3: 0.35, 4: 0.5, 5: 0.35, 6: 1.07, 7: 2.31, 8: 1.6, 9: 1.35, 10: 1.25, 11: 7.92, 12: 11.29}
best degree = 5; test RMSE = 0.288


In [5]:
# ✔ Self-check
check("validation error for degrees 1..12", lambda: sorted(val_errors) == list(range(1, 13)))
check("best_degree has the lowest validation error", lambda: best_degree == min(val_errors, key=val_errors.get))
check("a moderate degree wins (2 ≤ degree ≤ 8)", lambda: 2 <= best_degree <= 8)
check("test RMSE is reasonable (< 0.6)", lambda: test_error < 0.6)

✅ PASSED   validation error for degrees 1..12
✅ PASSED   best_degree has the lowest validation error
✅ PASSED   a moderate degree wins (2 ≤ degree ≤ 8)
✅ PASSED   test RMSE is reasonable (< 0.6)


True

**🧑‍🏫 Walkthrough.** This is the U-shape in action: validation error is large for degree 1 (underfit), dips at a moderate degree, and explodes for high degrees (overfit). Notice we picked the degree on **validation** data and reported on **test** data. With so few points, a different random split could pick a different degree: a good reason to prefer cross-validation (next problems!).

### 🧩 Problem 5.3: Rescue an overfit model with Ridge &nbsp; ⭐⭐

Using the same split as the previous problem (re-created below), compare, on the **validation** set, a **degree-12** polynomial with:

* **no regularization** (`poly_model(12)`) → store its validation RMSE in `plain_val`;
* **Ridge** for each `alpha` in `[1e-4, 1e-3, 1e-2, 0.03, 0.1, 0.3, 1, 3, 10]` (`poly_model(12, alpha=a)`) → store RMSEs in `ridge_val` (`{alpha: rmse}`).

Store the best alpha in `best_alpha`.

In [6]:
curve = load("noisy_curve.csv")
X_c, y_c = curve[["x"]], curve["y"]
X_train, X_rest, y_train, y_rest = train_test_split(X_c, y_c, train_size=20, random_state=3)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=3)

alphas = [1e-4, 1e-3, 1e-2, 0.03, 0.1, 0.3, 1, 3, 10]
plain_val = rmse(y_val, poly_model(12).fit(X_train, y_train).predict(X_val))
ridge_val = {a: rmse(y_val, poly_model(12, alpha=a).fit(X_train, y_train).predict(X_val)) for a in alphas}
best_alpha = min(ridge_val, key=ridge_val.get)
print(f"unregularized validation RMSE: {plain_val:.3f}")
print({a: round(e, 3) for a, e in ridge_val.items()})
print(f"best alpha = {best_alpha}  →  validation RMSE {ridge_val[best_alpha]:.3f}")

unregularized validation RMSE: 11.286
{0.0001: 2.412, 0.001: 1.12, 0.01: 0.403, 0.03: 0.477, 0.1: 0.806, 0.3: 1.012, 1: 0.853, 3: 0.556, 10: 0.465}
best alpha = 0.01  →  validation RMSE 0.403


In [7]:
# ✔ Self-check
check("one RMSE per alpha", lambda: sorted(ridge_val) == sorted(alphas))
check("best_alpha has the lowest validation RMSE", lambda: best_alpha == min(ridge_val, key=ridge_val.get))
check("regularization helps: best Ridge beats the unregularized model", lambda: ridge_val[best_alpha] < plain_val)
check("it helps a lot (at least 20% lower error)", lambda: ridge_val[best_alpha] < 0.8 * plain_val)

✅ PASSED   one RMSE per alpha
✅ PASSED   best_alpha has the lowest validation RMSE
✅ PASSED   regularization helps: best Ridge beats the unregularized model
✅ PASSED   it helps a lot (at least 20% lower error)


True

**🧑‍🏫 Walkthrough.** The model's *form* (degree 12) never changed: only the leash did. A moderate α gives a big drop in validation error; a huge α starts to underfit again (look at the last entries in the dictionary: error creeping back up). In a real project you'd pick α with cross-validation rather than one validation split.

### 🧩 Problem 5.4: Which features does Lasso keep? &nbsp; ⭐⭐

On `diabetes.csv` (features: every column except `disease_progression`), fit `Lasso(alpha=1)`.

Store the **number of coefficients that are exactly zero** in `n_zero`, and the **names of the features that survived** in a list `kept_features`.

In [8]:
diab = load("diabetes.csv")
X = diab.drop(columns="disease_progression")
y = diab["disease_progression"]

lasso = Lasso(alpha=1).fit(X, y)
n_zero = int((lasso.coef_ == 0).sum())
kept_features = list(X.columns[lasso.coef_ != 0])
print(f"{n_zero} of {X.shape[1]} coefficients are zero; kept: {kept_features}")

7 of 10 coefficients are zero; kept: ['bmi', 'bp', 's5']


In [9]:
# ✔ Self-check
_l = Lasso(alpha=1).fit(X, y)
check("count of zeroed coefficients is correct", lambda: n_zero == int((_l.coef_ == 0).sum()))
check("surviving feature names are correct", lambda: sorted(kept_features) == sorted(X.columns[_l.coef_ != 0]))
check("'bmi' survived (it is the strongest predictor)", lambda: "bmi" in kept_features)

✅ PASSED   count of zeroed coefficients is correct
✅ PASSED   surviving feature names are correct
✅ PASSED   'bmi' survived (it is the strongest predictor)


True

**🧑‍🏫 Walkthrough.** At α = 1, Lasso discards 7 of the 10 features, keeping a short, interpretable list (BMI, the `s5` serum measurement, and blood pressure). That is a model you can explain to a doctor in one sentence: a real benefit of L1 regularization. (This dataset's features are already standardised by scikit-learn; with your own data you must scale first.)

### 🧩 Problem 5.5: Cross-validate three models &nbsp; ⭐⭐

Compare these three models on `housing.csv` (target `price_k`) using **5-fold cross-validation** with `KFold(n_splits=5, shuffle=True, random_state=0)` and R² scoring:

* `"LinearRegression"`: `LinearRegression()`
* `"Ridge"`: `make_pipeline(StandardScaler(), Ridge(alpha=10))`
* `"KNN"`: `make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10))`

Store the **mean R²** of each in `cv_means` (keyed by the names above) and the name of the winner in `best_name`.

In [10]:
housing = load("housing.csv")
X, y = housing.drop(columns="price_k"), housing["price_k"]
kf = KFold(n_splits=5, shuffle=True, random_state=0)

models = {
    "LinearRegression": LinearRegression(),
    "Ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
    "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=10)),
}
cv_means = {name: cross_val_score(m, X, y, cv=kf, scoring="r2").mean() for name, m in models.items()}
best_name = max(cv_means, key=cv_means.get)
print({k: round(v, 4) for k, v in cv_means.items()}, "→", best_name)

{'LinearRegression': np.float64(0.9655), 'Ridge': np.float64(0.9651), 'KNN': np.float64(0.8569)} → LinearRegression


In [11]:
# ✔ Self-check
check("three models evaluated", lambda: sorted(cv_means) == ["KNN", "LinearRegression", "Ridge"])
check("winner is the model with the highest mean R²", lambda: best_name == max(cv_means, key=cv_means.get))
check("KNN is clearly the weakest here (R² < 0.9)", lambda: cv_means["KNN"] < 0.9)
check("linear models score above 0.95", lambda: cv_means["LinearRegression"] > 0.95 and cv_means["Ridge"] > 0.95)

✅ PASSED   three models evaluated
✅ PASSED   winner is the model with the highest mean R²
✅ PASSED   KNN is clearly the weakest here (R² < 0.9)
✅ PASSED   linear models score above 0.95


True

**🧑‍🏫 Walkthrough.** The two linear models are practically tied (their difference is far smaller than the fold-to-fold spread, so we'd call it a draw and prefer the simpler / more interpretable one), while KNN trails: this dataset's price is close to a linear function of the features, so a model that assumes linearity has the right bias. **A flexible model is not automatically a better model.**

### 🧩 Problem 5.6: Cross-validation from scratch &nbsp; ⭐⭐⭐

Implement `my_cross_val(model, X, y, k=5)` **without** `cross_val_score` or `KFold`.

* Split the row positions `0..n−1` into `k` consecutive folds with `np.array_split`.
* For each fold: `clone(model)`, fit on the other folds, compute `score` (R²) on the held-out fold.
* Return the **mean** of the k scores.

*(`X` is a DataFrame, `y` a Series; use `.iloc` with index arrays.)*

In [12]:
def my_cross_val(model, X, y, k=5):
    folds = np.array_split(np.arange(len(X)), k)
    scores = []
    for i, val_pos in enumerate(folds):
        train_pos = np.concatenate([f for j, f in enumerate(folds) if j != i])
        m = clone(model).fit(X.iloc[train_pos], y.iloc[train_pos])        # fresh, unfitted copy each fold
        scores.append(m.score(X.iloc[val_pos], y.iloc[val_pos]))
    return float(np.mean(scores))

_h = load("housing.csv")
print(round(my_cross_val(LinearRegression(), _h.drop(columns="price_k"), _h["price_k"]), 4))

0.9657


In [13]:
# ✔ Self-check
_h = load("housing.csv")
_X, _y = _h.drop(columns="price_k"), _h["price_k"]
_mine = my_cross_val(LinearRegression(), _X, _y, k=5)
_ref = cross_val_score(LinearRegression(), _X, _y, cv=KFold(n_splits=5), scoring="r2").mean()
check("matches sklearn's cross_val_score (no-shuffle KFold)", lambda: abs(_mine - _ref) < 1e-9)
check("works with k=10 too", lambda: 0.9 < my_cross_val(Ridge(alpha=1), _X, _y, k=10) < 1.0)

✅ PASSED   matches sklearn's cross_val_score (no-shuffle KFold)


✅ PASSED   works with k=10 too


True

**🧑‍🏫 Walkthrough.** The important detail is `clone(model)`: each fold needs a **fresh** model, otherwise knowledge from earlier folds would leak into later ones. That, plus "validate only on rows the model never trained on", is all cross-validation is.